In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")

FILE_NAMES = [
    "aisles.csv",
    "departments.csv",
    "products.csv",
    "orders.csv",
]

datasets = {
    name.replace(".csv", ""): pd.read_csv(DATA_DIR / name)
    for name in FILE_NAMES
}

In [2]:
def summarize_dataframe(name: str, df: pd.DataFrame) -> None:
    """Print basic exploratory information for a single dataframe."""
    separator = "=" * 60

    print(separator)
    print(f"Dataset: {name}")
    print(separator)

    print(f"\nShape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

    print("\nColumn info:")
    df.info()

    print("\nMissing values:")
    missing = df.isna().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_summary = pd.DataFrame({
        "missing_count": missing,
        "missing_pct": missing_pct,
    })
    missing_summary = missing_summary[missing_summary["missing_count"] > 0]

    if missing_summary.empty:
        print("No missing values.")
    else:
        print(missing_summary.to_string())

    print("\nFirst 5 rows:")
    display(df.head())
    print()

In [3]:
for dataset_name, dataframe in datasets.items():
    summarize_dataframe(dataset_name, dataframe)

Dataset: aisles

Shape: 134 rows x 2 columns

Column info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134 entries, 0 to 133
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   aisle_id  134 non-null    int64 
 1   aisle     134 non-null    object
dtypes: int64(1), object(1)
memory usage: 2.2+ KB

Missing values:
No missing values.

First 5 rows:


,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation



Dataset: departments

Shape: 21 rows x 2 columns

Column info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   department_id  21 non-null     int64 
 1   department     21 non-null     object
dtypes: int64(1), object(1)
memory usage: 464.0+ bytes

Missing values:
No missing values.

First 5 rows:


,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol



Dataset: products

Shape: 49,688 rows x 4 columns

Column info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49688 entries, 0 to 49687
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   product_id     49688 non-null  int64 
 1   product_name   49688 non-null  object
 2   aisle_id       49688 non-null  int64 
 3   department_id  49688 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 1.5+ MB

Missing values:
No missing values.

First 5 rows:


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13



Dataset: orders

Shape: 3,421,083 rows x 7 columns

Column info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3421083 entries, 0 to 3421082
Data columns (total 7 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int64  
 1   user_id                 int64  
 2   eval_set                object 
 3   order_number            int64  
 4   order_dow               int64  
 5   order_hour_of_day       int64  
 6   days_since_prior_order  float64
dtypes: float64(1), int64(5), object(1)
memory usage: 182.7+ MB

Missing values:
                        missing_count  missing_pct
days_since_prior_order         206209         6.03

First 5 rows:


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


In [5]:
#Hypothesis: user_id == days_since_prior_order
orders = datasets["orders"]

unique_user_count = orders["user_id"].nunique()
print(f"Unique user_id count in orders: {unique_user_count:,}")

nan_days_orders = orders[orders["days_since_prior_order"].isna()]
print(f"\nRows with NaN in days_since_prior_order: {len(nan_days_orders):,}")

all_first_orders = (nan_days_orders["order_number"] == 1).all()
print(f"\nHypothesis: all NaN rows have order_number == 1")
print(f"Result: {all_first_orders}")

if not all_first_orders:
    non_first_orders = nan_days_orders[nan_days_orders["order_number"] != 1]
    print(f"Counterexample count (order_number != 1): {len(non_first_orders):,}")
    print("\nSample counterexamples:")
    print(non_first_orders.head().to_string())
else:
    print("Confirmed: every row with NaN days_since_prior_order has order_number equal to 1.")

Unique user_id count in orders: 206,209

Rows with NaN in days_since_prior_order: 206,209

Hypothesis: all NaN rows have order_number == 1
Result: True
Confirmed: every row with NaN days_since_prior_order has order_number equal to 1.
